# Sentence Embedding Experiment with Google Colabolatory

# 0. Setup

In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from google.colab import drive, userdata
from sklearn.linear_model import LogisticRegression

drive.mount("/content/drive")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = 507

Mounted at /content/drive


# 1. Data Preparation

In [2]:
train = pd.read_parquet("drive/MyDrive/rq2/train.parquet")
val = pd.read_parquet("drive/MyDrive/rq2/validation.parquet")
test = pd.read_parquet("drive/MyDrive/rq2/test.parquet")

X_train_se = np.load("drive/MyDrive/rq2/train_embd.npy")
X_val_se = np.load("drive/MyDrive/rq2/validation_embd.npy")
X_test_se = np.load("drive/MyDrive/rq2/test_embd.npy")

# 2. Experiment

In [3]:
# AUC Score
def firm_auc_score(test_data, y_pred):
    df = test_data.copy()
    df["pred"] = y_pred
    records = []

    for (firm, rating), g in df.groupby(["firm_id", "rating"]):
        f_preds = np.array(g[g["former"] == 1]["pred"])
        c_preds = np.array(g[g["former"] == 0]["pred"])

        if len(f_preds) == 0 or len(c_preds) == 0:
            continue

        diff = f_preds[:, np.newaxis] - c_preds
        wins = (diff > 0).sum() + 0.5 * (diff == 0).sum()
        pairs = len(f_preds) * len(c_preds)

        records.append({"firm_id": firm, "wins": wins, "pairs": pairs})

    result = pd.DataFrame(records)
    wins_sum = result.groupby("firm_id")["wins"].sum()
    pairs_sum = result.groupby("firm_id")["pairs"].sum()
    firm_auc = wins_sum / pairs_sum
    return firm_auc

def calculate_auc_score(test_data, y_pred):
    return firm_auc_score(test_data, y_pred).mean()

In [4]:
def bootstrap_ci_rq2(firm_values, seed, n=2000):
    rng = np.random.default_rng(seed)
    x = firm_values.to_numpy()
    means = [rng.choice(x, size=len(x)).mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])

In [5]:
c_list = [0.01, 0.1, 1, 10]
auc_se_list = []

for c in c_list:
    lr_se = LogisticRegression(C=c, max_iter=1000, random_state=seed)
    model_se = lr_se.fit(X_train_se, train["former"])

    y_pred_se = model_se.predict_proba(X_val_se)[:, 1]
    auc_se = calculate_auc_score(val, y_pred_se)
    print(f"C: {c} Sentence Embedding AUC: {auc_se:.3f}\n")
    auc_se_list.append(auc_se)

print(f"================================\nFinal Sentence Embedding AUC: {max(auc_se_list):.3f}")

C: 0.01 Sentence Embedding AUC: 0.601

C: 0.1 Sentence Embedding AUC: 0.603

C: 1 Sentence Embedding AUC: 0.604

C: 10 Sentence Embedding AUC: 0.604

Final Sentence Embedding AUC: 0.604


In [6]:
lr_se = LogisticRegression(C=max(auc_se_list), max_iter=1000, random_state=seed)
model_se = lr_se.fit(X_train_se, train["former"])

y_pred_se = model_se.predict_proba(X_test_se)[:, 1]
firm_auc_se = firm_auc_score(test, y_pred_se)
lo_se, hi_se = bootstrap_ci_rq2(firm_auc_se, seed)
auc_se= calculate_auc_score(test, y_pred_se)

print(f"Sentence Embedding AUC: {auc_se:.3f} [{lo_se:.3f}, {hi_se:.3f}]")

Sentence Embedding AUC: 0.606 [0.602, 0.610]


In [20]:
firm_auc_tfidf = pd.read_parquet("drive/MyDrive/rq2/firm_auc_tfidf.parquet")["auc"]
lo_diff_2, hi_diff_2 = bootstrap_ci_rq2(firm_auc_se - firm_auc_tfidf, seed)

print(f"95% Confidence Interval for the difference between TF-IDF model and Sentence Embedding model is: [{lo_diff_2:.3f}, {hi_diff_2:.3f}]")

95% Confidence Interval for the difference between TF-IDF model and Sentence Embedding model is: [-0.031, -0.024]
